# LoRA Lens: four PEFT recipes on Colab

Thin Colab driver for **LoRA Lens**. Training logic lives in `src/smollm_lora/four_methods.py` and `scripts/train_four_methods.py` — this notebook only installs deps, mounts Drive, puts the repo on `sys.path`, and calls `run_four_methods`.

| Method | What changes | Config |
| --- | --- | --- |
| `original_lora` | baseline adapter | attention only, $r=8$, $\alpha=16$, scale $\alpha/r$, float base |
| `qlora` | frozen base storage | same adapter as above, 4-bit NF4 base |
| `all_linear` | which layers + rank | all linear layers, $r=16$, $\alpha=16$, scale $\alpha/r$ |
| `rslora` | scale only | same as `all_linear`, scale $\alpha/\sqrt{r}$ |

**Colab setup**
1. Runtime → Change runtime type → **T4 GPU** (or any CUDA GPU).
2. Upload the local `loratune` folder to Drive as `MyDrive/loratune` (needs `configs.py` and `src/smollm_lora/`).
3. Run every cell top to bottom.
4. Results land under Drive `loratune_experiments/four_methods/`.


## 0. Install, mount Drive, put the package on the path


In [ ]:
# GPU required for QLoRA (bitsandbytes).
!nvidia-smi

!pip -q install -U transformers datasets accelerate peft bitsandbytes matplotlib pandas tqdm


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/loratune_experiments/four_methods")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Source of truth is the local package. Upload Week10/loratune to Drive as
# MyDrive/loratune (or copy src/ + configs.py into that folder) before running.
# Do not rely on a GitHub clone until the package refactor is on the remote.
REPO = Path("/content/drive/MyDrive/loratune")
assert (REPO / "src" / "smollm_lora" / "four_methods.py").exists(), (
    "Upload the loratune repo to Drive at MyDrive/loratune so "
    "src/smollm_lora/four_methods.py is available."
)

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))
print("Package root:", REPO)
print("Saving results to:", DRIVE_ROOT)


## 1. Run all four methods via the shared package


In [ ]:
import torch
from configs import FOUR_METHODS, METHODS
from smollm_lora.four_methods import run_four_methods

assert torch.cuda.is_available(), "Enable a CUDA GPU runtime before continuing."
print("CUDA device:", torch.cuda.get_device_name(0))
print("Methods:", list(METHODS.keys()))

metrics = run_four_methods(
    DRIVE_ROOT,
    method_keys=list(METHODS.keys()),
    methods=METHODS,
    train_settings=FOUR_METHODS,
    make_figures=True,
)
metrics[["method", "trainable_params", "frozen_base_memory_mb", "test_loss", "train_seconds"]]


## 2. Pack a zip for download

Copy `four_methods_bundle.zip` out of Drive when the run is done. Locally, keep the recorded reference under `results/four_methods_bundle/` unless you intend to replace the article numbers.


In [ ]:
import shutil
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/loratune_experiments/four_methods_bundle")
shutil.make_archive(str(zip_path), "zip", root_dir=DRIVE_ROOT)
print("Bundle:", str(zip_path) + ".zip")
for path in sorted(DRIVE_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(DRIVE_ROOT), f"({path.stat().st_size} bytes)")


## Done

You should now have on Drive:

- `metrics.csv` — one row per method
- `train_curves.csv` — epoch losses
- `generations.csv` / `generations/*.json` — ten prompts × four methods
- `figures/*.png` — six comparison plots
- `adapters/<method>/` — PEFT adapters
- `run_config.json` — exact settings

Fill the blank `addressed` column in `generations.csv` if you want prompt scores in the write-up. Locally, `python scripts/train_four_methods.py` is the source of truth.
